# Machine Learning Pipelines

## What is a Pipeline?

A **Pipeline** (from `sklearn.pipeline`) is a way of **chaining multiple preprocessing steps and a final estimator (model) into a single object**, so that the entire workflow — transformations *and* model training/prediction — is executed in one sequential, reproducible unit.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('power_transform', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression())
])

pipe.fit(X_train, y_train)
predictions = pipe.predict(X_test)
```

Often combined with `ColumnTransformer` for mixed data types:

```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(), categorical_cols)
])

full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression())
])
```

## Why Pipelines Matter

1. **Prevents data leakage** — Each transformer's `fit()` (e.g., mean/std for scaling, λ for Box-Cox) is learned *only* on training data during `pipeline.fit()`, then correctly applied (`transform()` only, not re-fit) to test/validation data. Doing this manually is a common source of bugs where test data accidentally influences preprocessing.

2. **Reproducibility & cleanliness** — The entire workflow (transform → transform → model) becomes one object. No risk of forgetting a step, applying steps in the wrong order, or applying them inconsistently between training and inference.

3. **Simplifies cross-validation and hyperparameter tuning** — You can pass the whole pipeline into `GridSearchCV` or `cross_val_score`, and every fold correctly refits transformers on that fold's training data only:
```python
from sklearn.model_selection import GridSearchCV
params = {'model__C': [0.1, 1, 10]}
grid = GridSearchCV(full_pipeline, params, cv=5)
grid.fit(X_train, y_train)
```

4. **Production-readiness** — Once trained, the pipeline can be serialized (e.g., via `joblib` or `pickle`) as a single artifact, and deployed. New raw data just needs `pipeline.predict()` — no need to manually re-apply each preprocessing step at inference time.

5. **Cleaner, more maintainable code** — Instead of scattered preprocessing code, the whole flow is declared once, is easier to read, debug, and modify (e.g., swapping `StandardScaler` for `RobustScaler` is a one-line change).

**In short:** Pipelines turn a fragile, manually-ordered sequence of preprocessing + modeling steps into a single, leak-proof, reusable, and deployable object — which is why they're considered a best practice in any real-world ML workflow, not just a convenience.